# Plant Disease Detection Pipeline

This notebook demonstrates the complete machine learning pipeline for Plant Disease Detection, modularly structured for future IoT integration.

## 1. Import Modules and Verify GPU

In [ ]:
import torch
import os
import json
from pathlib import Path

# Check CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Dataset and Setup DataLoaders

In [ ]:
from src.dataset import get_dataloaders

dataset_dir = "PlantVillage"
batch_size = 32

train_loader, val_loader, test_loader, class_to_idx = get_dataloaders(
    dataset_dir=dataset_dir,
    batch_size=batch_size
)

print(f"Detected classes: {list(class_to_idx.keys())}")
# Save class indices mapping for future inference
with open("class_indices.json", "w") as f:
    json.dump(class_to_idx, f, indent=4)

## 3. Define and Load Model

In [ ]:
from src.model import get_plant_model

model_name = "mobilenet_v3_large"
num_classes = len(class_to_idx)

# Create model with frozen backbone
model = get_plant_model(
    model_name=model_name,
    num_classes=num_classes,
    pretrained=True,
    freeze_backbone=True
)

## 4. Train the Model

In [ ]:
from src.train import train_model

epochs = 2 # Setting to 2 for quick validation, increase for complete training
initial_lr = 1e-3

trained_model, history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=epochs,
    initial_lr=initial_lr,
    patience=5,
    device=device.type
)

# Save training history and model weights
with open("training_history.json", "w") as f:
    json.dump(history, f, indent=4)

torch.save(trained_model.state_dict(), "best_model.pth")
torch.save(trained_model.state_dict(), "final_model.pth")
print("Model weights saved successfully!")

## 5. View Loss and Accuracy curves

In [ ]:
from src.eval import plot_training_history

plot_training_history("training_history.json", output_dir=".")

## 6. Evaluate Model on Test Dataset

In [ ]:
from src.eval import evaluate_model_on_test

metrics = evaluate_model_on_test(
    model=trained_model,
    test_loader=test_loader,
    class_to_idx=class_to_idx,
    device=device.type,
    output_dir="."
)

## 7. Run Single Image Inference

In [ ]:
from src.predict import predict_single_image, load_class_indices

# Find a sample image dynamically from the dataset to test with
sample_class = list(class_to_idx.keys())[0]
sample_class_dir = Path(dataset_dir) / sample_class
sample_image_path = str(next(sample_class_dir.iterdir()))
print(f"Selected sample image for verification: {sample_image_path}")

idx_to_class = load_class_indices("class_indices.json")

predict_single_image(
    model=trained_model,
    image_path=sample_image_path,
    idx_to_class=idx_to_class,
    device=device.type,
    show_image=True
)